# Writing Style Profiler

Identify and characterize **my** writing voice from my LinkedIn posts so it can be reused to draft new posts that sound like me.

Pipeline:
1. Load original posts from the LinkedIn `Shares.csv` export (resharing/commentary on others is excluded).
2. Compute quantitative style metrics (length, rhythm, punctuation, emoji, hashtags, vocabulary richness).
3. Visualize the distributions and surface signature words/phrases.
4. Use the local Ollama model to synthesize a reusable **style guide** and save it to `outputs/`.

Conventions (zip discovery, original-vs-reshared split, Ollama model) mirror `webapp.py`.

In [ ]:
import json
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import ollama
import pandas as pd

# ── Config (same file webapp.py reads) ───────────────────────────────────────
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONFIG = json.loads((ROOT / "config" / "settings.json").read_text())

DATA_DIR = ROOT / CONFIG.get("data_dir", "data")
OUTPUT_DIR = ROOT / "outputs"
MODEL = CONFIG.get("model", "gemma4:latest")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Data dir : {DATA_DIR}")
print(f"Model    : {MODEL}")

## 1. Load original posts

`find_zip()` picks the latest `Complete_*.zip` export folder, exactly like `webapp.py`. A post is **original** when `SharedUrl` is empty; reshares of someone else's link carry their voice, not mine, so they are dropped. Empty/near-empty commentary is also filtered out.

In [ ]:
def find_zip() -> Path:
    zips = sorted(
        p for p in DATA_DIR.glob("Complete_*.zip") if not p.name.endswith(".zip.zip")
    )
    if not zips:
        raise FileNotFoundError(f"No Complete_*.zip export found in {DATA_DIR}")
    return zips[-1]


zip_path = find_zip()
df = pd.read_csv(
    zip_path / "Shares.csv",
    usecols=["Date", "SharedUrl", "ShareCommentary"],
)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["ShareCommentary"] = df["ShareCommentary"].fillna("").str.strip()
df["is_original"] = df["SharedUrl"].fillna("").str.strip() == ""

# Keep my own posts with real prose (drop one-liners / empty reshares).
posts = df[df["is_original"] & (df["ShareCommentary"].str.split().str.len() >= 5)].copy()
posts = posts.reset_index(drop=True)

print(f"Total share rows : {len(df)}")
print(f"Original posts   : {df['is_original'].sum()}")
print(f"Usable for style : {len(posts)}")
posts[["Date", "ShareCommentary"]].head()

## 2. Quantitative style metrics

Per-post features that together describe *mechanical* voice — how long things run, how the text breathes (line breaks/paragraphs), and the punctuation/emoji/hashtag habits that make a post recognizably mine.

In [ ]:
EMOJI_RE = re.compile(
    "[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F000-\U0001F0FF\u2190-\u21FF\u2B00-\u2BFF]"
)
SENT_RE = re.compile(r"[.!?]+(?:\s|$)")
WORD_RE = re.compile(r"[A-Za-z']+")


def metrics(text: str) -> dict:
    words = WORD_RE.findall(text)
    n_words = len(words)
    sentences = [s for s in SENT_RE.split(text) if s.strip()]
    n_sent = max(len(sentences), 1)
    lines = [ln for ln in text.splitlines() if ln.strip()]
    return {
        "chars": len(text),
        "words": n_words,
        "sentences": len(sentences),
        "words_per_sentence": n_words / n_sent,
        "avg_word_len": (sum(len(w) for w in words) / n_words) if n_words else 0,
        "lines": len(lines),
        "paragraphs": len([b for b in re.split(r"\n\s*\n", text) if b.strip()]),
        "questions": text.count("?"),
        "exclamations": text.count("!"),
        "ellipses": text.count("...") + text.count("\u2026"),
        "emojis": len(EMOJI_RE.findall(text)),
        "hashtags": len(re.findall(r"#\w+", text)),
        "mentions": len(re.findall(r"@\w+", text)),
        "caps_words": sum(1 for w in words if len(w) > 1 and w.isupper()),
        "ttr": (len({w.lower() for w in words}) / n_words) if n_words else 0,
    }


feat = posts["ShareCommentary"].apply(metrics).apply(pd.Series)
posts = pd.concat([posts, feat], axis=1)

summary = feat.describe().T[["mean", "50%", "std", "min", "max"]].round(2)
summary.columns = ["mean", "median", "std", "min", "max"]
summary

In [ ]:
# Headline profile (the numbers worth quoting to the LLM and to myself).
profile = {
    "posts_analyzed": int(len(posts)),
    "median_words": float(feat["words"].median()),
    "median_words_per_sentence": round(float(feat["words_per_sentence"].median()), 1),
    "median_paragraphs": float(feat["paragraphs"].median()),
    "pct_with_emoji": round(100 * (feat["emojis"] > 0).mean(), 1),
    "pct_with_hashtag": round(100 * (feat["hashtags"] > 0).mean(), 1),
    "pct_with_question": round(100 * (feat["questions"] > 0).mean(), 1),
    "emojis_per_post": round(float(feat["emojis"].mean()), 2),
    "hashtags_per_post": round(float(feat["hashtags"].mean()), 2),
    "vocab_richness_ttr": round(float(feat["ttr"].mean()), 3),
}
profile

## 3. Visualize the distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
specs = [
    ("words", "Words per post", "#0A66C2"),
    ("words_per_sentence", "Words per sentence", "#378FE9"),
    ("paragraphs", "Paragraphs per post", "#F5A623"),
    ("emojis", "Emojis per post", "#7A3FF2"),
]
for ax, (col, title, color) in zip(axes.ravel(), specs):
    ax.hist(feat[col], bins=20, color=color, edgecolor="white")
    ax.axvline(feat[col].median(), color="black", ls="--", lw=1,
               label=f"median {feat[col].median():.0f}")
    ax.set_title(title)
    ax.legend()
fig.suptitle("Mechanical writing-style distributions", fontsize=14)
fig.tight_layout()
plt.show()

## 4. Signature words & opening hooks

The vocabulary I lean on, and how I tend to *start* a post (the first line is the hook on LinkedIn).

In [ ]:
STOP = {
    "the", "a", "an", "and", "or", "but", "if", "to", "of", "in", "on", "for",
    "with", "as", "at", "by", "is", "are", "was", "were", "be", "been", "it",
    "this", "that", "these", "those", "i", "you", "we", "they", "he", "she",
    "my", "your", "our", "their", "me", "us", "them", "do", "does", "did",
    "have", "has", "had", "will", "would", "can", "could", "so", "not", "no",
    "from", "about", "just", "like", "all", "more", "out", "up", "how", "what",
    "when", "who", "which", "there", "here", "then", "than", "its", "im", "ive",
}

all_words = [
    w.lower()
    for text in posts["ShareCommentary"]
    for w in WORD_RE.findall(text)
    if len(w) > 2 and w.lower() not in STOP
]
top_words = Counter(all_words).most_common(25)

first_lines = [
    next((ln.strip() for ln in t.splitlines() if ln.strip()), "")
    for t in posts["ShareCommentary"]
]

print("Signature words:")
print(", ".join(f"{w} ({c})" for w, c in top_words))
print("\nSample opening hooks:")
for line in first_lines[:12]:
    print(" •", line[:90])

## 5. Synthesize the style guide (Ollama)

Feed the LLM the computed metrics plus a representative sample of full posts (longest = most voice-bearing), and ask it to write a reusable style guide: tone, structure, sentence rhythm, formatting habits, and dos/don'ts that another model could follow to draft in my voice.

In [ ]:
# Representative sample: a spread of longer, fuller posts (cap tokens sent).
sample = (
    posts.sort_values("words", ascending=False)
    .head(15)["ShareCommentary"]
    .tolist()
)
sample_block = "\n\n---\n\n".join(s[:1200] for s in sample)

prompt = f"""You are a writing coach analyzing one author's LinkedIn voice.

Measured style metrics across {profile['posts_analyzed']} of their original posts:
{json.dumps(profile, indent=2)}

Signature words: {', '.join(w for w, _ in top_words)}

Here are representative full posts:

{sample_block}

Write a concise STYLE GUIDE (markdown) that another writer or AI could follow to
draft new posts in this exact voice. Cover, with short concrete bullets:
1. Overall tone & personality
2. Typical structure (hook, body, close)
3. Sentence rhythm & length
4. Formatting habits (line breaks, lists, emoji, hashtags)
5. Recurring themes & vocabulary
6. Dos and Don'ts
Ground every claim in the metrics or sample above. Do not invent facts about the author."""

resp = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}])
style_guide = resp.message.content.strip()
print(style_guide)

## 6. Save the style profile

Persist both the machine-readable metrics and the LLM style guide to `outputs/` so the webapp or a drafting step can load them.

In [ ]:
(OUTPUT_DIR / "writing_style_metrics.json").write_text(
    json.dumps(
        {
            "profile": profile,
            "top_words": top_words,
            "summary": summary.to_dict(),
        },
        indent=2,
    )
)

report = f"""# My LinkedIn Writing Style

_Generated from {profile['posts_analyzed']} original posts in `{zip_path.name}`._

## Quick profile

| Metric | Value |
|---|---|
""" + "\n".join(f"| {k} | {v} |" for k, v in profile.items()) + f"""

## Signature words

{', '.join(w for w, _ in top_words)}

## Style guide

{style_guide}
"""

out = OUTPUT_DIR / "writing_style.md"
out.write_text(report)
print(f"Saved {out} and writing_style_metrics.json")